In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6 gradio

In [ ]:
from google.colab import userdata
from IPython.display import Markdown, display, update_display
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc
import gradio as gr

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
models = {
    "LLAMA" : "meta-llama/Llama-3.2-1B-Instruct",
    "PHI" : "microsoft/Phi-4-mini-instruct",
    "GEMMA" : "google/gemma-3-270m-it",
    "QWEN" : "Qwen/Qwen3-4B-Instruct-2507",
    "DEEPSEEK" : "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
}

In [ ]:
# Quantization Config - this allows us to load the model into memory and use less memory

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
def generate(model, messages, quant=False, max_new_tokens=2000):
  tokenizer = AutoTokenizer.from_pretrained(model)
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")
  streamer = TextStreamer(tokenizer)
  if quant:
    model_instance = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to("cuda")
  else:
    model_instance = AutoModelForCausalLM.from_pretrained(model).to("cuda")

  outputs = model_instance.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)
  # Modified line: Added skip_special_tokens=True
  response = tokenizer.decode(outputs[0], skip_special_tokens=True)
  display(Markdown(response))

  # Clean up memory
  del model_instance
  del tokenizer
  del input_ids
  del attention_mask
  gc.collect()
  torch.cuda.empty_cache()


In [ ]:
def stream_answer(prompt, model):
    system_prompt = "You are a helpful assistant in help users generating data. Please provide information in a clear and concise table format whenever possible."
    messages = [{'role': 'system', 'content': system_prompt}, {'role': 'user', 'content': prompt}]
    model = models[model]
    result = generate(model, messages, quant=True)
    return result

In [ ]:
stream_answer("Give me a dataset of high school students", "LLAMA")

In [ ]:
message_input = gr.Textbox(label="Enter the dataset you want to generate and select the open source model you want to use", info="Enter the kind of dataset you want to generate:", lines=7)
model_selector = gr.Dropdown(list(models.keys()), label="Select model", value="LLAMA")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_answer,
    title="Dataset Generator",
    inputs=[message_input, model_selector],
    outputs=[message_output],
    flagging_mode="never",
)
view.launch()